# Feature Engineering

## Introduction

Feature engineering is the process of refining the data you have and creating new, meaningful features from it. Good features are the difference between a mediocre model and a great one; garbage in, garbage out.

In this lab, you'll practice the core techniques: handling categorical data with encoding, creating new features, and scaling features. We'll use a housing dataset to make things concrete.

Run the cells below to import libraries and load the dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

In [ ]:
data = {
    "address":        ["123 Shattuck", "456 Telegraph", "789 Durant", "101 Bancroft", "202 Piedmont", "303 College", "404 Haste", "505 Dwight"],
    "size_sqft":      [1200, 850, 2100, 650, 1750, 1100, 3000, 950],
    "num_bedrooms":   [3, 2, 4, 1, 3, 2, 5, 2],
    "num_bathrooms":  [2, 1, 3, 1, 2, 1, 4, 1],
    "year_built":     [1995, 2010, 1980, 2018, 2003, 1972, 2015, 2008],
    "neighborhood":   ["Downtown", "Midtown", "Uptown", "Downtown", "Suburbs", "Suburbs", "Uptown", "Midtown"],
    "condition":      ["Good", "Excellent", "Fair", "Excellent", "Good", "Poor", "Good", "Fair"],
    "has_garage":     [True, False, True, False, True, False, True, False],
    "price":          [850000, 620000, 1250000, 480000, 975000, 390000, 1800000, 540000]
}

df = pd.DataFrame(data)
df

# Part 1: Beginner (Required)

### Q1: One-Hot Encoding

Machine learning models need numbers, not text. The `neighborhood` column has four categories (`Downtown`, `Midtown`, `Uptown`, `Suburbs`) with no natural ordering between them. This is a great case for **one-hot encoding**, which creates a separate binary column for each category.

Run the cell below to see what one-hot encoding looks like on the `neighborhood` column.

In [ ]:
neighborhood_encoded = pd.get_dummies(df["neighborhood"])
neighborhood_encoded

Now attach these new columns to the original dataframe and drop the original `neighborhood` column. Save the result back to `df` and display it.

*Hint: use [pd.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) with axis=1*

In [ ]:
df = ...

**Written Question**: The `condition` column also has text categories: `Poor`, `Fair`, `Good`, `Excellent`. Should we use one-hot encoding or label encoding for this column, and why?

> *Your answer here:*

### Q2: Label Encoding

We now apply **label encoding** to the `condition` column. Label encoding assigns a unique integer to each category, and works best when categories have a meaningful order (ordinal data).

The code is already written out for you. Make sure to answer the written question that follows.

In [ ]:
encoder = LabelEncoder()

df["condition_encoded"] = encoder.fit_transform(df["condition"])

# Print the mapping
print("Encoding:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))
df[["condition", "condition_encoded"]]

**Written Question 1** Concisely explain the code above to the best of your ability. The [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) may help here (it is important to understand it as you will use it later!).

> *Your answer here:*

**Written Question 2**: Look at the mapping that was printed. Does the encoding correctly reflect the order `Poor < Fair < Good < Excellent`? If not, what problem could this cause for a model?

> *Your answer here:*

### Q3: Creating New Features

One of the most impactful things you can do as a data scientist is engineer new features using domain knowledge. Raw columns often don't tell the full story, but combinations of them might.

**Create three new columns in `df`:**
- `total_rooms`: total number of bedrooms and bathrooms combined
- `house_age`: how many years old the house is (use 2025 as the current year)
- `price_per_sqft`: price divided by size in square feet

Then display only the `address` column alongside your three new ones.

In [ ]:
# Create your three new features here
df['total_rooms'] = 
df['house_age'] = 
df['price_per_sqft'] = 

# Display the new columns here 
df[...]

Now let's see how these new features correlate with price. Run the cell below.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feature in zip(axes, ["total_rooms", "house_age", "price_per_sqft"]):
    sns.scatterplot(data=df, x=feature, y="price", ax=ax)
    ax.set_title(f"{feature} vs price")

plt.tight_layout()
plt.show()

**Written Question**: Which of the three new features appears to have the strongest relationship with price? What does the direction of that relationship tell you?

> *Your answer here:*

### Q4: Feature Scaling

Look at the `size_sqft` and `num_bedrooms` columns. One is in the thousands, the other in single digits. A model using raw values would incorrectly treat `size_sqft` as ~1000x more important simply because its numbers are bigger. **Feature scaling** fixes this.

Two common approaches:
- **Normalization (Min-Max Scaling)**: scales values to the range [0, 1]
- **Standardization (Z-score Scaling)**: scales values to have mean 0 and standard deviation 1

The scalers work similarly to `LabelEncoder`: call `.fit_transform()` on your data and wrap the result in a `pd.DataFrame` to get it back as a table.

**Complete the code below** to apply both scalers to `cols_to_scale` and store the results in `df_normalized` and `df_standardized`. Name the new columns by appending `_norm` and `_std` to the original column names.

In [ ]:
cols_to_scale = ["size_sqft", "num_bedrooms"]

min_max  = MinMaxScaler()
standard = StandardScaler()

df_normalized   = ...
df_standardized = ...

pd.concat([df[cols_to_scale], df_normalized, df_standardized], axis=1)

## Part 2: Advanced (Optional)

### Q1: Which Features Actually Matter?

Feature selection is about deciding which features are worth keeping. One quick method: look at the **correlation** between each numeric feature and the target variable (`price`). Features with a correlation close to 0 are often not useful.

**Your task** — write code to:
1. Select only numeric columns from `df`
2. Compute the full correlation matrix with `.corr()`, then pull out just the `"price"` column and drop the `"price"` row (so you're left with feature-to-price correlations only). [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) may help but you can also do this just by seeing what the .corr function outputs below.
3. Plot a horizontal bar chart of the result

Save the sorted correlations as `correlations` — you'll need it in Q2.

In [ ]:
# Step 1: select numeric columns
numeric_df = 

# Step 2: compute correlations with price
correlations = numeric_df.corr()._______ #there should be more code after the .corr() call

# Step 3: plot
correlations.plot(_______)


**Written Question**: Based on the correlation values, which 3 features would you keep if you could only use 3? Which feature would you drop first, and why?

> *Your answer here:*

### Q2: Building the Best Feature Set

Now put it all together. Using `correlations` from Q1, write code to:
1. Find all features where the absolute correlation with `price` is greater than `0.5`
2. Build `df_final` containing only those features plus `price` (from the numeric columns of `df`)
3. Display `df_final`

Then run the heatmap cell to visualize relationships among your selected features.

In [ ]:
# Find features with |correlation| > 0.5
strong_features = ...

# Build df_final with those features + price
df_final = numeric_df[____]
df_final

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_final.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Selected Features")
plt.tight_layout()
plt.show()

**Written Question**: Look at the heatmap. Do any two non-price features correlate strongly with *each other*? Why might having two highly correlated features in your model be a problem?

> *Your answer here:*

Congrats, you've finished this notebook!